# Milestone 4 · Vérification du plan de Wissam et des gains

Le plan retenu du Milestone 3 est celui de Wissam, figé au commit **a0e8aba**. Il utilise les lignes préparées avec imputation à la capacité maximale lorsque nécessaire, et **RG06 assouplie** : une ligne peut rejoindre n’importe quel départ de son couple dans sa fenêtre J−7 à J. Les besoins hors du jeu préparé restent hors du calcul.

Ce notebook rapproche les données, les affectations et les camions. Il recalcule la référence et les coûts, puis mesure ce qui change pour les usines. Depuis le dépôt, récupérer l’historique avec git fetch origin. Le notebook utilise pandas, NumPy, SciPy et holidays.

In [1]:
from pathlib import Path
from types import ModuleType
from io import BytesIO
import subprocess
import sys
import numpy as np
import pandas as pd

commit = 'a0e8aba'
cwd = Path.cwd().resolve()
root = next(p for p in (cwd, *cwd.parents) if (p / 'milestone2/planification_testbed').is_dir())

def git_file(path):
    command = ['git', '-c', f'safe.directory={root.as_posix()}', '-C', str(root), 'show', f'{commit}:{path}']
    return subprocess.check_output(command)

def csv(name):
    return pd.read_csv(BytesIO(git_file(f'data/{name}.csv')))

source = csv('consolidated_dataset_max')
aff = csv('candidat_max_sortie2_affectations')
camions = csv('candidat_max_sortie2_camions')
profils = csv('candidat_max_sortie1_profils')
gains = csv('candidat_max_sortie3_saving')
cles = ['supplier_id', 'plant']
ligne = cles + ['date', 'material', 'unloading_point']

## 1. Refaire les contrôles de couverture et de capacité

On contrôle les valeurs par ligne avant de vérifier les totaux par camion. Cette étape détecte notamment un doublon, une quantité changée ou un camion attribué au mauvais couple.

In [2]:
colonnes = ligne + ['schedule_quantity', 'weight', 'volume']
attendu = source[colonnes].sort_values(ligne).reset_index(drop=True)
observe = aff.rename(columns={'need_date': 'date'})[colonnes].sort_values(ligne).reset_index(drop=True)
pd.testing.assert_frame_equal(attendu, observe, check_dtype=False)
assert len(aff) == aff['line_id'].nunique()
assert not camions['truck_id'].duplicated().any()

recolle = aff.merge(camions, on='truck_id', how='left', validate='m:1', suffixes=('', '_truck'), indicator=True)
assert recolle['_merge'].eq('both').all()
for detail, camion in [('supplier_id', 'supplier_id_truck'), ('plant', 'plant_truck'), ('delivery_date', 'date')]:
    assert recolle[detail].eq(recolle[camion]).all()
charges = aff.groupby('truck_id')[['weight', 'volume']].sum().sort_index()
pd.testing.assert_frame_equal(charges, camions.set_index('truck_id')[['weight', 'volume']].sort_index())
assert charges['weight'].le(24500 + 1e-8).all()
assert charges['volume'].le(110.4 + 1e-8).all()
print(f'{len(aff)} besoins uniques, {len(camions)} camions, quantités et charges conservées.')

4389 besoins uniques, 760 camions, quantités et charges conservées.


## 2. Clarifier les règles de calendrier

Wissam prévoit des départs de démarrage pour les premiers besoins qui précèdent le premier créneau du profil. Les jours du profil sont contrôlés sur les départs réguliers. RG06 stricte est mesurée à part pour rendre visible la différence entre les deux lectures.

In [3]:
import holidays

date_depart = pd.to_datetime(aff['delivery_date'])
date_besoin = pd.to_datetime(aff['need_date'])
avance = (date_besoin - date_depart).dt.days
feries = holidays.country_holidays('DE', years=sorted(set(date_depart.dt.year)))
assert avance.between(0, 7).all() and avance.eq(aff['advance_days']).all()
assert date_depart.dt.dayofweek.lt(5).all()
assert not date_depart.dt.date.isin(feries).any()

departs = aff[cles + ['delivery_date']].drop_duplicates().sort_values('delivery_date')
departs['suivant'] = departs.groupby(cles)['delivery_date'].shift(-1)
controle = aff.merge(departs, on=cles + ['delivery_date'], validate='m:1')
ecarts = pd.to_datetime(controle['need_date']).ge(pd.to_datetime(controle['suivant']))
print(f'Avance de {avance.min()} à {avance.max()} jours, aucun jour non ouvré.')
print(f'RG06 stricte : {ecarts.sum()} lignes en écart sur {controle.loc[ecarts, cles].drop_duplicates().shape[0]} couples.')
print(f'Démarrage : {aff.profile_day.isna().sum()} lignes hors créneaux réguliers.')

Avance de 0 à 7 jours, aucun jour non ouvré.
RG06 stricte : 142 lignes en écart sur 51 couples.
Démarrage : 92 lignes hors créneaux réguliers.


In [4]:
reguliers = aff.dropna(subset=['profile_day'])[cles + ['profile_day']].drop_duplicates().copy()
jour = pd.to_datetime(reguliers['profile_day'])
reguliers['jour'] = jour.dt.dayofweek
reguliers['semaine'] = jour - pd.to_timedelta(reguliers['jour'], unit='D')
fixes = reguliers.groupby(cles)['jour'].nunique()
pic = reguliers.groupby(cles + ['semaine']).size().groupby(level=[0, 1]).max()
publie = profils.set_index(cles)['profile'].sort_index()
assert fixes.sort_index().eq(publie).all()
assert pic.sort_index().eq(publie).all()
print(f'RG05 : {len(publie)} profils correspondent à leur semaine régulière la plus chargée.')

RG05 : 180 profils correspondent à leur semaine régulière la plus chargée.


## 3. Comparer les coûts et le remplissage

La référence retenue rassemble les besoins au jour prévu, par fournisseur × usine. Elle charge les lignes entières et applique à chaque camion la même grille qu’au plan de Wissam. Les euros sont des coûts calculés et les économies sont l’écart entre ces deux scénarios.

In [5]:
planner = ModuleType('wissam_m4')
sys.modules[planner.__name__] = planner
exec(compile(git_file('src/delivery_schedule_optimization/planner.py'), 'planner.py', 'exec'), planner.__dict__)
config = planner.PlannerConfig()
grille = pd.read_csv(BytesIO(git_file('docs/rate_sheet.csv'))).set_index('km_up_to').astype(float)
borne = lambda x: np.inf if str(x).startswith('>') else float(x)
grille.index, grille.columns = grille.index.map(borne), grille.columns.map(borne)
tarif = planner.Tariff(grille.sort_index().sort_index(axis=1), config)

avant = csv('baseline_deliveries_max')
source['date'] = pd.to_datetime(source['date'])
avant['date'] = pd.to_datetime(avant['date'])
reference, camions_avant = planner.current_situation(source, avant, tarif, config)
assert np.allclose(camions['cost'], tarif(camions['distance'], camions['weight']))
assert np.isclose(reference['cost'].sum(), gains['cost_before'].sum())

In [6]:
couts = gains.groupby('plant')[['trucks_before', 'trucks_after', 'cost_before', 'cost_after', 'saving']].sum()
couts.loc['TOTAL'] = couts.sum()
couts['saving_pct'] = 100 * couts['saving'] / couts['cost_before']
assert np.isclose(couts.loc['TOTAL', 'cost_after'], camions['cost'].sum())
assert gains['saving'].ge(-0.005).all()
print(couts.round(2).to_string())

         trucks_before  trucks_after  cost_before  cost_after    saving  saving_pct
plant                                                                              
PLT_001          540.0         222.0     45200.68    27979.34  17221.35       38.10
PLT_002          452.0         219.0     45004.43    28789.34  16215.09       36.03
PLT_003          193.0         107.0     27931.81    19387.52   8544.29       30.59
PLT_004          556.0         212.0     58587.68    34387.76  24199.92       41.31
TOTAL           1741.0         760.0    176724.61   110543.95  66180.66       37.45


In [7]:
avant, apres = camions_avant.copy(), camions.copy()
for table in (avant, apres):
    table['poids_utilise_pct'] = 100 * table['weight'] / config.truck_kg
    table['volume_utilise_pct'] = 100 * table['volume'] / config.truck_m3
mesures = ['weight', 'volume', 'poids_utilise_pct', 'volume_utilise_pct']
distribution = pd.concat({nom: table[mesures].quantile([.25, .5, .75, .9]).T
                          for nom, table in [('référence', avant), ('plan Wissam', apres)]})
print(distribution.round(2).to_string())
for nom, table in [('référence', avant), ('plan Wissam', apres)]:
    pic_global = table.groupby('date').size().max()
    pics_usines = table.groupby(['plant', 'date']).size().groupby('plant').max().to_dict()
    print(f'{nom} | pic global {pic_global} | pics par usine {pics_usines}')

                                  0.25     0.50     0.75      0.90
référence   weight              142.00   365.50  1097.70   6018.00
            volume                0.43     1.20     2.40      7.05
            poids_utilise_pct     0.58     1.49     4.48     24.56
            volume_utilise_pct    0.39     1.09     2.17      6.38
plan Wissam weight              288.32  1053.50  7202.07  24500.00
            volume                0.89     2.44     9.16    110.40
            poids_utilise_pct     1.18     4.30    29.40    100.00
            volume_utilise_pct    0.80     2.21     8.29    100.00
référence | pic global 118 | pics par usine {'PLT_001': 31, 'PLT_002': 54, 'PLT_003': 16, 'PLT_004': 24}
plan Wissam | pic global 68 | pics par usine {'PLT_001': 18, 'PLT_002': 39, 'PLT_003': 9, 'PLT_004': 18}


## 4. Cas limites et lecture des résultats

La prise de décision repose sur un périmètre préparé. On rapproche les besoins bruts pour compter les lignes imputées, les besoins du 1er janvier et la ligne lourde sortie en amont. Ces lignes sont importantes pour comprendre le gain annoncé et la portée de la couverture.

In [8]:
brut = pd.read_csv(BytesIO(git_file('docs/released_schedule.csv')))
brut['date'] = pd.to_datetime(brut['date'])
positifs = brut.loc[brut['schedule_quantity'].gt(0)].copy()
dans_horizon = positifs.loc[positifs['date'].lt(pd.Timestamp('2027-01-01'))]
joint = source[ligne].merge(dans_horizon[ligne + ['weight', 'volume']], on=ligne, validate='1:1')
manque = joint[['weight', 'volume']].fillna(0).le(0).any(axis=1)
exclus = dans_horizon.merge(source[ligne], on=ligne, how='left', indicator=True).query('_merge == "left_only"')
print(f'{manque.sum()} lignes planifiées avec au moins une charge inconnue dans la source.')
print(f'{len(positifs)-len(dans_horizon)} besoins du 1er janvier hors horizon.')
print(exclus[ligne + ['schedule_quantity', 'weight', 'volume']].to_string(index=False))

134 lignes planifiées avec au moins une charge inconnue dans la source.
379 besoins du 1er janvier hors horizon.
supplier_id   plant       date   material unloading_point  schedule_quantity  weight  volume
  SUP_00074 PLT_003 2026-12-15 MAT_001164        ULP_0003             5670.0 26905.2    0.84


In [9]:
# Bornes inclusives et palier tarifaire au-delà de 4 tonnes.
assert tarif(121, 4000) < tarif(121, 4000.01)
assert tarif(121, 4000.01) == tarif(121, 24500)
# Un exemple où le chargement glouton ouvre un camion de trop.
kg = np.array([14700, 12250, 7350, 4900, 4900, 4900])
charge_gloutonne = planner.pack(kg, np.ones(6), config)
alternative = [[0, 3, 4], [1, 2, 5]]
assert len(charge_gloutonne) == 3
assert all(kg[lot].sum() <= config.truck_kg for lot in alternative)
print('Tarif : seuil de 4 t vérifié. Cas synthétique : 3 camions gloutons, 2 faisables.')

Tarif : seuil de 4 t vérifié. Cas synthétique : 3 camions gloutons, 2 faisables.


## 5. Situer les économies par rapport aux imputations

On regroupe les couples selon qu’ils contiennent au moins une charge ou une distance imputée. Ce découpage décrit les économies du plan publié. Il ne mesure pas l’effet causal de l’imputation : pour cela il faudrait recalculer deux plans sur un périmètre comparable.

In [10]:
distances = pd.read_csv(BytesIO(git_file('docs/distances.csv')))
distances_connues = distances.loc[distances['distance'].gt(0), 'supplier_id']
joint['imputation'] = manque | ~joint['supplier_id'].isin(distances_connues)
indicateur = joint.groupby(cles)['imputation'].any()
ventilation = gains.set_index(cles).join(indicateur, validate='1:1')
assert ventilation['imputation'].notna().all()
for statut, groupe in ventilation.groupby('imputation'):
    etiquette = 'avec' if statut else 'sans'
    print(f"Couples {etiquette} imputation : {len(groupe)} | économies {groupe['saving'].sum():.2f} €")
print(f"{joint['imputation'].sum()} lignes avec une charge ou une distance imputée.")

Couples sans imputation : 150 | économies 55097.58 €
Couples avec imputation : 30 | économies 11083.08 €
138 lignes avec une charge ou une distance imputée.


## 6. Bilan et suite du Milestone 4

- Les objectifs de coût et de nombre de camions se vérifient sur les résultats publiés.
- RG06 est assouplie par choix d’équipe. Le compteur strict reste visible dans ce notebook.
- Le scénario retenu seul a pris **46,11 s de bout en bout sur 8 processus**, génération des candidats, lissage, contrôles et exports compris. Les 15 contrôles du moteur passent et les affectations correspondent ligne par ligne à l’export publié. [Détail du chronométrage](chrono_wissam.json).
- Les valeurs imputées et la référence reconstruite restent à distinguer des mesures physiques et des factures. Le contre-exemple de chargement établit une limite de la méthode, sans remettre en cause la faisabilité de ses affectations publiées.